# Build the corpus

Rebuilds the two corpus files the pipeline reads, fetching both through the MediaWiki API:

| File | Source | Selection |
|---|---|---|
| `documents/wikipedia_pages.json` | English Wikipedia | 33 fixed titles: the series, its titles and spin-offs, the consoles they released on, and the best-selling games list |
| `documents/mh_wiki_monsters.json` | [monsterhunterwiki.org](https://monsterhunterwiki.org) | every page in `Category:Large Monsters` and `Category:Small Monsters` |

Both files map page title to plain text. Run the notebook top to bottom. It takes a few minutes, mostly the
delay between Monster Hunter Wiki requests. Each save reports what changed against the existing file before
overwriting it.

Afterwards run `python -m rag.pipeline` from the repository root. Only chunks whose text changed are
re-embedded, and chunks the new corpus no longer produces are deleted from the store.

In [ ]:
import json
import re
import time
from pathlib import Path

import requests
import wikipediaapi
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# Works whether the kernel starts in notebooks/ or the repository root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT_DIR = ROOT / "documents"

# Wikimedia's API policy asks for a descriptive User-Agent with contact details
USER_AGENT = "MH-RAG/0.1 (https://github.com/DhLYa/RAG_Pipeline_Testing)"
HEADERS = {"User-Agent": USER_AGENT}

In [ ]:
def save_corpus(pages: dict[str, str], path: Path) -> None:
    """Report what changed against the existing file, then overwrite it."""
    if path.exists():
        with open(path, encoding="utf-8") as f:
            old = json.load(f)
        added = sorted(pages.keys() - old.keys())
        removed = sorted(old.keys() - pages.keys())
        changed = [t for t in pages.keys() & old.keys() if pages[t] != old[t]]
        print(f"vs existing {path.name}: {len(added)} added, {len(removed)} removed, {len(changed)} changed")
        if added:
            print(f"  added: {added[:10]}")
        if removed:
            print(f"  removed: {removed[:10]}")

    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)

    total = sum(len(v) for v in pages.values())
    print(f"Saved {len(pages)} pages, {total:,} characters, to {path}")

## 1. Wikipedia

The title list is fixed rather than taken from a category, because the consoles and the best-selling games
list are not in `Category:Monster_Hunter`. Titles are kept exactly as first written (underscores and all):
each one becomes its chunks' `source` metadata and is hashed into their IDs, so renaming a title would force
every chunk from that page to be re-embedded. MediaWiki resolves them to the canonical articles.

Trailing sections holding citations and links are dropped by section title. `page.text` renders headings as
bare lines, so searching the text for `== References` markup, as the original notebook did, matches nothing.

In [ ]:
WIKIPEDIA_TITLES = [
    # Series, main titles and spin-offs
    "Monster_Hunter",
    "Monster_Hunter_(video_game)",
    "Monster_Hunter_2",
    "Monster_Hunter_4",
    "Monster_Hunter_Diary",
    "Monster_Hunter_Dynamic_Hunting",
    "Monster_Hunter_Freedom",
    "Monster_Hunter_Freedom_2",
    "Monster_Hunter_Freedom_Unite",
    "Monster_Hunter_Frontier_Z",
    "Monster_Hunter_Generations",
    "Monster_Hunter_Now",
    "Monster_Hunter_Outlanders",
    "Monster_Hunter_Portable_3rd",
    "Monster_Hunter_Rise",
    "Monster_Hunter_Stories",
    "Monster_Hunter_Stories_2: Wings_of_Ruin",
    "Monster_Hunter_Stories_3: Twisted_Reflection",
    "Monster_Hunter_Tri",
    "Monster_Hunter_Wilds",
    "Monster_Hunter_World: Iceborne",
    "Monster_Hunter: Legends_of_the_Guild",
    "Monster_Hunter: World",
    # Consoles the series released on
    "Nintendo_3DS",
    "Nintendo_Switch",
    "PlayStation",
    "PlayStation_(console)",
    "PlayStation_2",
    "PlayStation_3",
    "PlayStation_4",
    "PlayStation_Portable",
    "PlayStation_Vita",
    # Sales context
    "List_of_best-selling_video_games",
]

# Top-level sections that hold citations and links rather than prose
DROP_SECTIONS = {
    "References", "Notes", "Citations", "Footnotes", "Sources", "Bibliography",
    "External links", "See also", "Further reading",
}

wiki = wikipediaapi.Wikipedia(user_agent=USER_AGENT, language="en")


def article_text(page) -> str:
    """page.text without the DROP_SECTIONS. Assembled the same way wikipediaapi
    builds page.text, so the text that is kept is formatted identically."""
    text = page.summary
    if text:
        text += "\n\n"
    for section in page.sections:
        if section.title not in DROP_SECTIONS:
            text += section.full_text(level=2)
    return text.strip()

In [ ]:
wikipedia_pages, missing = {}, []
for title in tqdm(WIKIPEDIA_TITLES, unit="page"):
    page = wiki.page(title)
    if page.exists():
        wikipedia_pages[title] = article_text(page)
    else:
        missing.append(title)

# A missing title means an article was renamed or deleted; fix the list rather than save a smaller corpus
if missing:
    raise RuntimeError(f"Wikipedia pages not found: {missing}")

save_corpus(wikipedia_pages, OUT_DIR / "wikipedia_pages.json")

## 2. Monster Hunter Wiki

Pages are requested as rendered HTML and reduced to prose:

- **Infobox first.** `table.monster-game-info` is read into label-value lines (classification, elements,
  weaknesses, ...) before the remaining tables are stripped. Element and status values are rendered as icons
  whose names live on the parent link, so they are recovered from the link rather than the image.
- **Headings only with content.** A heading is kept only when prose follows it. Sections whose content was in
  the stripped tables or galleries would otherwise leave runs of bare headings that become empty chunks.
- **Noise removed.** Navigation bars, galleries, figures, reference lists and MediaWiki citation errors.
- **Stubs skipped.** Pages whose prose, excluding the infobox, is under `MIN_BODY_CHARS`, or whose only content
  is the legend explaining the stat tables.

In [ ]:
MH_API = "https://monsterhunterwiki.org/api.php"
CATEGORIES = ["Category:Large Monsters", "Category:Small Monsters"]

MIN_PAGE_CHARS = 400
MIN_BODY_CHARS = 300          # prose alone, excluding the infobox
REQUEST_DELAY = 0.3           # seconds between page requests

INFOBOX_SELECTOR = "table.monster-game-info"

# This wiki uses <h1> for section headings, not <h2>
HEADING_TAGS = {"h1", "h2", "h3", "h4", "h5", "h6"}
BLOCK_TAGS = ["p", "h1", "h2", "h3", "h4", "h5", "h6", "li", "dd", "dt", "blockquote"]

DROP_SELECTORS = [
    "table",
    "div.navbarheader",
    "div.navbar-green",
    "div.toc",
    "sup.reference",
    "span.mw-editsection",
    "figure",
    "figcaption",
    "div.thumb",
    "div.gallery",
    "div.mw-references-wrap",
    ".mw-ext-cite-error",      # "Cite error: <ref> tags exist for..."
    "span.error",
]

SKIP_NS = ("File:", "Help:", "Category:", "Template:", "Special:", "Talk:")

# Pages whose only content is the legend explaining the stat tables we strip.
# Their monster data lived in those tables, so nothing usable remains.
BOILERPLATE_MARKERS = (
    "The following information is for",
    "Part - the name of the part",
    "the effectiveness of Cutting, Blunt, and Shot",
)

In [ ]:
def category_members(category, namespace=0):
    """All article titles in a category, following API continuation."""
    titles = []
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": category,
        "cmlimit": "max",
        "cmnamespace": namespace,
        "format": "json",
    }
    while True:
        r = requests.get(MH_API, params=params, headers=HEADERS, timeout=30)
        r.raise_for_status()
        data = r.json()
        titles.extend(m["title"] for m in data["query"]["categorymembers"])
        if "continue" not in data:
            break
        params.update(data["continue"])
        time.sleep(0.2)
    return titles


def tidy(text):
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([.,;:!?)\]])", r"\1", text)
    text = re.sub(r"([(\[])\s+", r"\1", text)
    text = re.sub(r",\s*\)", ")", text)          # "(Dragon,)" -> "(Dragon)"
    text = re.sub(r"\(\s*\)", "", text)          # parens left empty by icons
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip(" ,;")


def table_to_text(table):
    """Turn a wikitable into label-value lines. Handles both vertical
    (th | td per row) and horizontal (header row then data row) layouts."""
    parts = []
    pending_headers = None
    for row in table.find_all("tr"):
        headers = row.find_all("th", recursive=False)
        cells = row.find_all("td", recursive=False)
        if headers and cells:
            for h, c in zip(headers, cells):
                label = tidy(h.get_text(" ", strip=True))
                value = tidy(c.get_text(" ", strip=True))
                if label and value:
                    parts.append(f"{label}: {value}")
            pending_headers = None
        elif headers and not cells:
            pending_headers = [tidy(h.get_text(" ", strip=True)) for h in headers]
        elif cells and pending_headers:
            values = [tidy(c.get_text(" ", strip=True)) for c in cells]
            for label, value in zip(pending_headers, values):
                if label and value:
                    parts.append(f"{label}: {value}")
            pending_headers = None
    return "\n".join(parts)


def infobox_text(soup):
    table = soup.select_one(INFOBOX_SELECTOR)
    if not table:
        return ""

    for sup in table.find_all("sup"):
        sup.decompose()

    # element and status values are icons; the name lives on the parent link
    for img in table.find_all("img"):
        a = img.find_parent("a")
        href = a.get("href", "") if a else ""
        if not a or any(href.startswith(f"/wiki/{ns}") for ns in SKIP_NS):
            img.replace_with("")                 # renders, help links, etc.
            continue
        label = a.get("title") or href.rsplit("/", 1)[-1].replace("_", " ")
        label = re.sub(r"\s+Element$", "", label)
        img.replace_with(f" {label}, " if label else "")

    return table_to_text(table)


def body_text(soup):
    """Emit prose blocks, keeping a heading only when real content follows it.

    Sections whose content was in stripped tables or galleries otherwise leave
    long runs of consecutive headings with nothing beneath them.
    """
    blocks = []
    pending = []
    for el in soup.find_all(BLOCK_TAGS):
        text = tidy(el.get_text(" ", strip=True))
        if not text:
            continue
        if el.name in HEADING_TAGS:
            pending = [text]          # nearest heading only; earlier ones were unbacked
            continue
        if pending:
            blocks.extend(pending)
            pending = []
        blocks.append(text)
    return "\n".join(blocks)


def fetch_page_text(title):
    r = requests.get(MH_API, params={
        "action": "parse",
        "page": title,
        "prop": "text",
        "format": "json",
        "formatversion": 2,
    }, headers=HEADERS, timeout=30)

    if r.status_code != 200:
        return None
    data = r.json()
    if "parse" not in data:
        return None

    soup = BeautifulSoup(data["parse"]["text"], "html.parser")

    facts = infobox_text(soup)          # extract before decomposing
    for selector in DROP_SELECTORS:
        for tag in soup.select(selector):
            tag.decompose()
    body = body_text(soup)

    # Judge the stub check on prose alone: a page with a populated infobox but
    # no descriptive text is still a stub, and checking the combined length
    # would let it through on the strength of the infobox.
    if any(m in body for m in BOILERPLATE_MARKERS):
        return None
    if len(body) < MIN_BODY_CHARS:
        return None

    return "\n\n".join(p for p in [facts, body] if p).strip()


def fetch_all(titles):
    pages, failed = {}, []
    bar = tqdm(titles, unit="page", desc="fetching")
    for title in bar:
        try:
            text = fetch_page_text(title)
        except Exception as e:
            failed.append((title, type(e).__name__))
            continue

        if text and len(text) >= MIN_PAGE_CHARS:
            pages[title] = text
        else:
            failed.append((title, "stub, empty, or too short"))

        chars = sum(len(v) for v in pages.values())
        bar.set_postfix_str(f"{title[:24]} | {len(pages)} kept | {chars:,} chars")
        time.sleep(REQUEST_DELAY)

    bar.close()
    if failed:
        print(f"\n{len(failed)} failed or skipped, e.g. {failed[:5]}")
    return pages

In [ ]:
members = {c: category_members(c) for c in CATEGORIES}
mh_titles = sorted(set().union(*members.values()))

print(", ".join(f"{c.removeprefix('Category:')}: {len(t)}" for c, t in members.items()),
      f"| unique total: {len(mh_titles)}")

Optional: preview a few cleaned pages before committing to the full fetch.

In [ ]:
for title, text in fetch_all(mh_titles[:3]).items():
    print(f"\n--- {title} ({len(text)} chars) ---")
    print(text[:700])

In [ ]:
mh_pages = fetch_all(mh_titles)
print(f"\nKept {len(mh_pages)} of {len(mh_titles)} pages.")

save_corpus(mh_pages, OUT_DIR / "mh_wiki_monsters.json")

## 3. Checks

In [ ]:
# Pages without an infobox lose the structured attributes (classification, elements, weaknesses)
no_infobox = [t for t, x in mh_pages.items() if "Classification:" not in x]
print(f"{len(no_infobox)} of {len(mh_pages)} Monster Hunter Wiki pages lack an infobox")
print(no_infobox[:15])

# Neither corpus should still contain the sections stripped above
leftover = [t for t, x in wikipedia_pages.items()
            if re.search(r"^(References|External links)$", x, flags=re.MULTILINE)]
print(f"Wikipedia pages still containing reference sections: {leftover or 'none'}")